In [1]:
# run_k_times.py
import os
import argparse
import random
import numpy as np
import torch
import torch.nn as nn

from torch_geometric.loader import DataLoader

# Local imports from your project
from baselineGCN import GCN, do_epoch, device
from loadMoleculeData import load_datasets
from model_configs import ConfigManager

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (/Users/n/.pyenv/versions/3.10.0/envs/baseml/lib/python3.10/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-lightning de

In [14]:

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    return do_epoch(loader, training=False, model=model, criterion=criterion)

def train_one_run(dataset_dict, config_manager: ConfigManager, seed: int, save_dir: str):
    set_seed(seed)
    name = dataset_dict['name']
    config = config_manager.get_config(name)

    train_loader = DataLoader(dataset_dict['train_set'], batch_size=config.batch_size, shuffle=True)
    val_loader   = DataLoader(dataset_dict['val_set'],   batch_size=config.batch_size, shuffle=False)
    test_loader  = DataLoader(dataset_dict['test_set'],  batch_size=config.batch_size, shuffle=False)

    in_channels = dataset_dict['train_set'].dataset.num_features
    model = GCN(in_channels=in_channels, config=config).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_val_auc = -float("inf")
    best_ckpt_path = os.path.join(save_dir, f"baselineGCN_{name}_seed{seed}.pt")
    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(1, config.epochs + 1):
        train_loss, train_auc = do_epoch(train_loader, training=True, model=model,
                                         optimizer=optimizer, criterion=criterion)
        val_loss,   val_auc   = evaluate(model, val_loader, criterion)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), best_ckpt_path)

        if epoch % 5 == 0 or epoch == 1:
            print(f"[{name} | seed {seed}] Epoch {epoch:03d} | "
                  f"Train loss {train_loss:.4f} AUC {train_auc:.3f} | "
                  f"Val AUC {val_auc:.3f}")

    ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(ckpt)
    _, test_auc = evaluate(model, test_loader, criterion)

    print(f"[{name} | seed {seed}] Best Val AUC {best_val_auc:.3f} | Test AUC @best-val {test_auc:.3f}")
    return best_val_auc, test_auc

def run_k_times(dataset_names=None, runs: int = 5, seeds=None, save_dir="../models", data_root="data/moleculenet"):
    """
    For each dataset (with its ConfigManager config), train K times and report mean ± std AUC.
    """
    config_manager = ConfigManager()
    config_manager.initialize_default_configs()

    datasets = load_datasets(names=dataset_names, root=data_root)
    if not datasets:
        raise ValueError(f"No datasets loaded. Check dataset names and that data exists at: {data_root}")


    # If seeds not provided, make deterministic, distinct seeds
    if seeds is None:
        # Start from a fixed base to make whole experiment reproducible
        base = 42
        seeds = [base + i for i in range(runs)]
    else:
        assert len(seeds) == runs, "Length of seeds must equal number of runs."

    summary = {}  # dataset_name -> dict

    for d in datasets:
        name = d['name']
        val_aucs = []
        test_aucs = []
        print(f"\n=== Running {name} for {runs} runs ===")
        for s in seeds:
            best_val_auc, test_auc = train_one_run(d, config_manager, s, save_dir)
            val_aucs.append(best_val_auc)
            test_aucs.append(test_auc)

        val_mean, val_std = float(np.mean(val_aucs)), float(np.std(val_aucs, ddof=1)) if len(val_aucs) > 1 else 0.0
        test_mean, test_std = float(np.mean(test_aucs)), float(np.std(test_aucs, ddof=1)) if len(test_aucs) > 1 else 0.0

        summary[name] = {
            "val_aucs": val_aucs,
            "test_aucs": test_aucs,
            "val_mean": val_mean, "val_std": val_std,
            "test_mean": test_mean, "test_std": test_std
        }

        print(f"\n>>> {name} results over {runs} runs")
        print(f"Val AUCs : {['{:.3f}'.format(x) for x in val_aucs]}")
        print(f"Test AUCs: {['{:.3f}'.format(x) for x in test_aucs]}")
        print(f"Val  AUC  mean±std: {val_mean:.3f} ± {val_std:.3f}")
        print(f"Test AUC  mean±std: {test_mean:.3f} ± {test_std:.3f}")

    return summary






In [20]:
DATA_ROOT = (Path.cwd().parent / "data" / "moleculenet")
results = run_k_times(dataset_names=["BBBP", "BACE", "HIV"], data_root=DATA_ROOT)



=== Running BBBP for 5 runs ===
[BBBP | seed 42] Epoch 001 | Train loss 0.5664 AUC 0.501 | Val AUC 0.809
[BBBP | seed 42] Epoch 005 | Train loss 0.4593 AUC 0.613 | Val AUC 0.850
[BBBP | seed 42] Epoch 010 | Train loss 0.4150 AUC 0.740 | Val AUC 0.850
[BBBP | seed 42] Epoch 015 | Train loss 0.4090 AUC 0.742 | Val AUC 0.868
[BBBP | seed 42] Epoch 020 | Train loss 0.4043 AUC 0.749 | Val AUC 0.876
[BBBP | seed 42] Epoch 025 | Train loss 0.3777 AUC 0.784 | Val AUC 0.884
[BBBP | seed 42] Epoch 030 | Train loss 0.3929 AUC 0.762 | Val AUC 0.885
[BBBP | seed 42] Epoch 035 | Train loss 0.3936 AUC 0.747 | Val AUC 0.883
[BBBP | seed 42] Epoch 040 | Train loss 0.3830 AUC 0.768 | Val AUC 0.895
[BBBP | seed 42] Epoch 045 | Train loss 0.3604 AUC 0.802 | Val AUC 0.894
[BBBP | seed 42] Epoch 050 | Train loss 0.3729 AUC 0.776 | Val AUC 0.904
[BBBP | seed 42] Epoch 055 | Train loss 0.3626 AUC 0.794 | Val AUC 0.908
[BBBP | seed 42] Epoch 060 | Train loss 0.3446 AUC 0.820 | Val AUC 0.918
[BBBP | seed 42] E